In [24]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import make_pipeline
import nltk
from nltk.corpus import stopwords
import warnings
warnings.filterwarnings('ignore') # Hides unnecessary warning messages

# 1. Download the NLTK stop words list (only needs to run once)
try:
    nltk.data.find('corpora/stopwords')
except LookupError:
    print("Downloading NLTK stopwords...")
    nltk.download('stopwords', quiet=True)

print("✅ Libraries imported successfully!")

# 2. Create a tiny, fake dataset of restaurant reviews
data = {
    'review': [
        "The food was absolutely amazing and the service was great.",
        "Terrible experience. The soup was cold and the waiter was rude.",
        "I loved the pasta, it tasted very authentic.",
        "Worst meal of my life. Will never go back."
    ],
    'sentiment': ['positive', 'negative', 'positive', 'negative']
}
df = pd.DataFrame(data)
print("\n✅ Dataset created:")
print(df)

# 3. Create a basic NLP Pipeline (TF-IDF Feature Extraction + Naive Bayes Model)
# We use the english stop words we downloaded earlier
stop_words_list = stopwords.words('english')

model = make_pipeline(
    TfidfVectorizer(stop_words=stop_words_list), 
    MultinomialNB()
)

# 4. Train the model on our fake dataset
model.fit(df['review'], df['sentiment'])
print("\n✅ Model trained successfully!")

# 5. Test the model with a brand new review it hasn't seen before
test_review = ["The waiter was awful but the food was okay."]
prediction = model.predict(test_review)

print(f"\n🔮 Prediction Test:")
print(f"Review: '{test_review[0]}'")
print(f"Predicted Sentiment: {prediction[0].upper()}")

✅ Libraries imported successfully!

✅ Dataset created:
                                              review sentiment
0  The food was absolutely amazing and the servic...  positive
1  Terrible experience. The soup was cold and the...  negative
2       I loved the pasta, it tasted very authentic.  positive
3         Worst meal of my life. Will never go back.  negative

✅ Model trained successfully!

🔮 Prediction Test:
Review: 'The waiter was awful but the food was okay.'
Predicted Sentiment: POSITIVE


In [25]:
%pip install pandas scikit-learn nltk

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [26]:
# Load the TSV dataset into a Pandas DataFrame
df = pd.read_csv('Restaurant_Reviews.tsv', sep='\t')

# SYSTEM VALIDATION: Drop any rows where the text or label is missing
df = df.dropna(subset=['Review', 'Liked'])

# Display the first 5 rows to see what columns you have
df.head()

,Review,Liked
0,Wow... Loved this place.,1
1,Crust is not good.,0
2,Not tasty and the texture was just nasty.,0
3,Stopped by during the late May bank holiday of...,1
4,The selection on the menu was great and so wer...,1


In [27]:
import nltk

# Download the tools needed for tokenization and lemmatization
nltk.download('punkt')      # For breaking sentences into words (Tokenization)
nltk.download('wordnet')    # For finding the root of a word (Lemmatization)
nltk.download('omw-1.4')    # Supporting data for wordnet
nltk.download('stopwords')  # The list of useless words (e.g., 'the', 'is')


[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\hengy\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\hengy\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     C:\Users\hengy\AppData\Roaming\nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\hengy\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [28]:
import re
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer

# 1. We are using ONLY the Lemmatizer this time
lemmatizer = WordNetLemmatizer()
stop_words = set(stopwords.words('english'))

# 2. Keep negative words
words_to_keep = ['not', 'no', 'nor', 'doesn', 'isn', 'wasn', 'shouldn', 'wouldn', 'couldn', 'won', 'can', 'didn', 'don', 'aren', 'haven', 'hasn', 'hadn']
for word in words_to_keep:
    stop_words.discard(word)

def clean_review(text):
    text = str(text).lower()
    text = re.sub(r'[^a-z\s]', '', text)
    
    # Requirement: Tokenization
    words = word_tokenize(text)
    
    # Requirement: Lemmatization (Keeps words looking normal!)
    cleaned_words = [lemmatizer.lemmatize(word) for word in words if word not in stop_words]
    
    return ' '.join(cleaned_words)

In [29]:
# This creates a brand new column called 'cleaned_review' with our processed text
df['cleaned_review'] = df['Review'].apply(clean_review)

# Let's look at the original vs the cleaned version side-by-side
df[['Review', 'cleaned_review']].head()

,Review,cleaned_review
0,Wow... Loved this place.,wow loved place
1,Crust is not good.,crust not good
2,Not tasty and the texture was just nasty.,not tasty texture nasty
3,Stopped by during the late May bank holiday of...,stopped late may bank holiday rick steve recom...
4,The selection on the menu was great and so wer...,selection menu great price


In [30]:
import nltk
# Download the newer required tokenizer files
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\hengy\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

In [31]:
from sklearn.feature_extraction.text import TfidfVectorizer

# 1. Initialize the TF-IDF Vectorizer
# We limit it to the top 2500 most common words to keep the model efficient
vectorizer = TfidfVectorizer(max_features=2500)

# 2. Convert the cleaned text into a matrix of numbers (Features / X)
X = vectorizer.fit_transform(df['cleaned_review']).toarray()

# 3. Define our target variable (Labels / y)
# IMPORTANT: Double-check that your column for positive/negative ratings is named 'Liked'
y = df['Liked'] 

print(f"Features (X) shape: {X.shape}")
print(f"Labels (y) shape: {y.shape}")

Features (X) shape: (1000, 1811)
Labels (y) shape: (1000,)


In [32]:
from sklearn.model_selection import train_test_split

# Split the data into 80% training and 20% testing
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42)

print(f"Training data shape: {X_train.shape}")
print(f"Testing data shape: {X_test.shape}")

Training data shape: (800, 1811)
Testing data shape: (200, 1811)


In [33]:
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, classification_report

# 1. INITIALIZE: Set up the Support Vector Machine
# The 'linear' kernel is the standard choice for text classification
svm_model = SVC(kernel='linear', random_state=42)

# 2. TRAIN: Teach the AI using your 80% training data
print("Training the SVM model...")
svm_model.fit(X_train, y_train)

# 3. TEST: Give the AI its final exam using the 20% hidden data
print("Testing the model...")
svm_predictions = svm_model.predict(X_test)

# 4. EVALUATE: Calculate and print the final grade
accuracy = accuracy_score(y_test, svm_predictions)
print("-" * 30)
print(f"✅ Final Accuracy: {accuracy * 100:.2f}%\n")
print("Detailed Classification Report:")
print(classification_report(y_test, svm_predictions))

Training the SVM model...
Testing the model...
------------------------------
✅ Final Accuracy: 82.50%

Detailed Classification Report:
              precision    recall  f1-score   support

           0       0.77      0.90      0.83        96
           1       0.89      0.76      0.82       104

    accuracy                           0.82       200
   macro avg       0.83      0.83      0.82       200
weighted avg       0.83      0.82      0.82       200



In [34]:
import pandas as pd

# 1. Grab the original review text for our test samples
test_indices = y_test.index

# 2. Build a clear table matching text to predictions
results_df = pd.DataFrame({
    'Review': df.loc[test_indices, 'Review'],
    'Actual_Sentiment': y_test.values,
    'Predicted_Sentiment': svm_predictions
})

# 3. Map 1 and 0 to actual human words
label_map = {1: 'Positive', 0: 'Negative'}
results_df['Actual_Sentiment'] = results_df['Actual_Sentiment'].map(label_map)
results_df['Predicted_Sentiment'] = results_df['Predicted_Sentiment'].map(label_map)

# 4. Show the first 10 reviews with their predicted outcomes
results_df.head(10)

,Review,Actual_Sentiment,Predicted_Sentiment
521,If you haven't gone here GO NOW!,Positive,Negative
737,Try them in the airport to experience some tas...,Positive,Positive
740,The restaurant is very clean and has a family ...,Positive,Positive
660,"I personally love the hummus, pita, baklava, f...",Positive,Positive
411,"Come hungry, leave happy and stuffed!",Positive,Positive
678,It's a great place and I highly recommend it.,Positive,Positive
626,Best of luck to the rude and non-customer serv...,Negative,Negative
513,Reasonably priced also!,Positive,Positive
859,Worst food/service I've had in a while.,Negative,Negative
136,I had a seriously solid breakfast here.,Positive,Positive


In [35]:
# ---------------------------------------------------------
# ADVANCED PROTOTYPE: Custom Review Sentiment Predictor
# ---------------------------------------------------------

def test_my_ai(custom_review):
    # --- UI/UX: Organized layout for the console ---
    print("\n" + "=" * 55)
    print("LIVE SENTIMENT ANALYSIS SYSTEM")
    print("=" * 55)
    
    # --- PROGRAMMING RUBRIC: Thorough Validation ---
    if not custom_review or custom_review.strip() == "":
        print("VALIDATION ERROR: Input cannot be empty.")
        print("Please provide a valid text review to analyze.")
        print("=" * 55)
        return # Stop the function from running with bad data
        
    # --- PROGRAMMING RUBRIC: Exception Handling ---
    try:
        # 1. Clean and vectorize (matching dense format)
        vectorized_text = vectorizer.transform([custom_review]).toarray()
        
        # 2. SVM Model Prediction
        prediction = svm_model.predict(vectorized_text)
        
        # 3. UI/UX: Clean, well-organized output layout
        print(f" Target Text  : \"{custom_review}\"")
        if prediction[0] == 1:
            print(" Prediction   : POSITIVE")
        else:
            print(" Prediction   : NEGATIVE")
            
    except Exception as e:
        # Catch any unexpected system errors gracefully
        print(f"SYSTEM ERROR: An unexpected fault occurred: {e}")
        
    print("=" * 55)

# --- Live Demonstration ---
# Test 1: Standard positive input
test_my_ai("The food was absolutely delicious and the service was amazing!")

# Test 2: Standard negative input
test_my_ai("I waited an hour for my meal and it was completely cold. Never coming back.")

# Test 3: Triggering the validation (Testing an empty string)
test_my_ai("    ")


LIVE SENTIMENT ANALYSIS SYSTEM
 Target Text  : "The food was absolutely delicious and the service was amazing!"
 Prediction   : POSITIVE

LIVE SENTIMENT ANALYSIS SYSTEM
 Target Text  : "I waited an hour for my meal and it was completely cold. Never coming back."
 Prediction   : NEGATIVE

LIVE SENTIMENT ANALYSIS SYSTEM
VALIDATION ERROR: Input cannot be empty.
Please provide a valid text review to analyze.
